In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

In [9]:
## Path to database code folder

path_to_database_python_code_folder = '/content/drive/Shared drives/CAGEO/Database_python_code/'

os.chdir(path_to_database_python_code_folder)

In [10]:
from codes.new_workbook import database as db

In [11]:
# 🔹 Define your base directory here
BASE_DIR = '/content/drive/Shareddrives/CAGEO/Database/'

# 🔹 Get only subfolders
folders = [f for f in os.listdir(BASE_DIR)
           if os.path.isdir(os.path.join(BASE_DIR, f))]

if not folders:
    print("No folders found in the directory.")
else:
    dropdown = widgets.Dropdown(
        options=folders,
        description='Select Folder:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='50%')
    )

    button = widgets.Button(description="Change Directory")
    output = widgets.Output()

    def on_button_click(b):
        with output:
            output.clear_output()
            selected_folder = os.path.join(BASE_DIR, dropdown.value)

            # 🔹 Change working directory
            os.chdir(selected_folder)

            print(f"Changed working directory to:\n{os.getcwd()}")

    button.on_click(on_button_click)

    display(dropdown, button, output)

Dropdown(description='Select Folder:', layout=Layout(width='50%'), options=('Carrickittle', 'Gortdrum', 'Canad…

Button(description='Change Directory', style=ButtonStyle())

Output()

# Creating the excel workbook

In [12]:
# -----------------------------
# 1) Get drillholes (folders in current directory)
# -----------------------------
cwd = os.getcwd()
drillholes = sorted([f for f in os.listdir(cwd)
                     if os.path.isdir(os.path.join(cwd, f))])

if not drillholes:
    raise RuntimeError(f"No folders found in current directory: {cwd}")

# -----------------------------
# 2) Available properties
# -----------------------------
ALL_PROPERTIES = ['den','ip_res','cond','sus','v_p','v_s','gamma','xrf']

# -----------------------------
# 3) Widgets
# -----------------------------

# Drillhole dropdown
drillhole_dd = widgets.Dropdown(
    options=drillholes,
    description="Drillhole:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

# Property checkboxes
property_checkboxes = [
    widgets.Checkbox(value=False, description=prop)
    for prop in ALL_PROPERTIES
]

prop_box = widgets.VBox(property_checkboxes)

run_btn = widgets.Button(description="Run", button_style='success')
output = widgets.Output()

# -----------------------------
# 4) Run Logic
# -----------------------------
def on_run_clicked(b):
    with output:
        clear_output()

        drillhole = drillhole_dd.value

        # Collect selected properties
        property_list = [cb.description for cb in property_checkboxes if cb.value]

        if not property_list:
            print("❌ Please select at least one property.")
            return

        print(f"✅ Selected drillhole: {drillhole}")
        print(f"✅ Selected properties: {property_list}")

        # ---- Your original workflow ----
        excelfile = db(drillhole)

        excelfile.createworkbook()
        excelfile.add_sheets(property_list)
        excelfile.copy_col_for_all_sheets(property_list)
        excelfile.add_cal(property_list)

        for prop in property_list:
            excelfile.add_data(prop)

        print("🎉 Processing complete.")

run_btn.on_click(on_run_clicked)

display(drillhole_dd, widgets.Label("Select Properties:"), prop_box, run_btn, output)

Dropdown(description='Drillhole:', layout=Layout(width='50%'), options=('DDH24-3368-03', 'FM-468-1', 'G-2529-4…

Label(value='Select Properties:')

Button(button_style='success', description='Run', style=ButtonStyle())

Output()